# Datetime Variables in Python and Pandas

Date and time data appears across almost every industry — retail sales, financial markets, healthcare records, web traffic, and more. Whether you are predicting how many products will sell next week or analysing when customer activity peaks, you need to handle dates correctly before you can extract any insight from them.

The challenge is that dates arrive in many formats: as strings like `'2021-05-14'`, as integer timestamps, or even as inconsistent free text. Until you convert them into a proper **datetime type**, you cannot do arithmetic, sort, or extract features like day of the week or month of the year.

This notebook covers two layers of datetime handling: first, Python's built-in `datetime` module for working with individual date and time objects; then, Pandas tools for working with entire columns of dates in a DataFrame.

## Learning Objectives

By the end of this notebook you will be able to:

1. Create `date`, `time`, and `datetime` objects using Python's `datetime` module

2. Parse a date string into a `datetime` object using `strptime()`

3. Format a `datetime` object back to a string using `strftime()`

4. Calculate the duration between two dates using `timedelta`

5. Convert a DataFrame column from string to datetime using `pd.to_datetime()`

6. Extract date components (day, month, year, weekday) using the `.dt` accessor

7. Generate a sequence of dates using `pd.date_range()`

## Setup

In [1]:
# Import pandas and the datetime module components
import pandas as pd
from datetime import datetime, date, time, timedelta

---

## Part 1: Python's `datetime` Module

Python's standard library includes the `datetime` module, which provides classes for representing dates and times as objects — not just strings.

The key distinction: a **string** like `'2021-05-14'` is just text. A **`date` object** understands what it represents — you can compare it, do arithmetic on it, and extract its parts.

### The `date` class

In [2]:
# Create a date object and a plain string representing the same day
d1 = date(2021, 5, 14)   # a proper date object
d2 = '2021-05-14'        # just a string

# The output looks identical — but the types are very different
print(d1)
print(d2)

2021-05-14
2021-05-14


In [3]:
# Confirm the type difference
print(type(d1))  # <class 'datetime.date'>
print(type(d2))  # <class 'str'>

<class 'datetime.date'>
<class 'str'>


In [4]:
# A date object lets you access its components as attributes
d3 = date.today()
print('Date of today:', d3)
print('Day:  ', d3.day)
print('Month:', d3.month)
print('Year: ', d3.year)

Date of today: 2026-05-12
Day:   12
Month: 5
Year:  2026


### The `time` class

In [5]:
# Create a time object: time(hour, minute, second, microsecond)
t1 = time(13, 20, 13, 40)  # a proper time object
t2 = '13:20:13.000040'     # just a string

print(t1)
print(t2)
print(type(t1))
print(type(t2))

13:20:13.000040
13:20:13.000040
<class 'datetime.time'>
<class 'str'>


In [6]:
# Access time components via attributes
print('Hour:       ', t1.hour)
print('Minute:     ', t1.minute)
print('Second:     ', t1.second)
print('Microsecond:', t1.microsecond)

Hour:        13
Minute:      20
Second:      13
Microsecond: 40


### The `datetime` class

The `datetime` class combines both date and time into a single object. It is the most commonly used class when working with timestamps.

In [7]:
# Create a datetime object: datetime(year, month, day, hour, minute, second, microsecond)
dt1 = datetime(2021, 5, 14, 11, 20, 30, 40)
print(dt1)
print(type(dt1))

2021-05-14 11:20:30.000040
<class 'datetime.datetime'>


In [8]:
# Get the current local date and time
dt2 = datetime.now()
dt2

datetime.datetime(2026, 5, 12, 11, 58, 57, 84275)

### Extracting the day of the week

One of the most useful features for data science is extracting the **day of the week** — for example, to capture the difference between weekday and weekend sales patterns.

- **`.weekday()`** returns an integer from 0 (Monday) to 6 (Sunday)
- **`.isoweekday()`** returns an integer from 1 (Monday) to 7 (Sunday)

In [9]:
# Extract the day of the week from today's date
dt_now = datetime.now()

# .weekday(): Monday = 0, Sunday = 6
print('weekday():', dt_now.weekday())

# .isoweekday(): Monday = 1, Sunday = 7
print('isoweekday():', dt_now.isoweekday())

weekday(): 1
isoweekday(): 2


### Week of the year

In [10]:
# .isocalendar() returns a named tuple with year, week number, and weekday
dt_now = datetime.now()
print('Full calendar info:', dt_now.isocalendar())
print('Week of year:      ', dt_now.isocalendar()[1])

Full calendar info: datetime.IsoCalendarDate(year=2026, week=20, weekday=2)
Week of year:       20


---

## Part 2: Parsing and Formatting Dates

Dates in datasets are almost always stored as **strings**. Before you can do any date arithmetic or extract components, you need to parse them into `datetime` objects.

Python provides two complementary methods:
- **`strptime()`** — parses a **string** into a `datetime` object ("string parse time")
- **`strftime()`** — formats a `datetime` object back into a **string** ("string format time")

Both use **format codes** to describe the structure of the date string. The most common codes are:

| Code | Meaning | Example |
|------|---------|--------|
| `%Y` | 4-digit year | `2021` |
| `%m` | Zero-padded month | `05` |
| `%d` | Zero-padded day | `14` |
| `%H` | Hour (24-hour) | `13` |
| `%M` | Minute | `20` |
| `%S` | Second | `30` |
| `%B` | Full month name | `April` |
| `%A` | Full weekday name | `Tuesday` |

See the [full format code reference](https://docs.python.org/3/library/datetime.html#strftime-and-strptime-format-codes) for all options.

### `strptime` — string to datetime

In [11]:
# Parse a date string with a custom format
date_str = '22 April, 2020 13:20:13'
d1 = datetime.strptime(date_str, '%d %B, %Y %H:%M:%S')
print(d1)
print(type(d1))

2020-04-22 13:20:13
<class 'datetime.datetime'>


In [12]:
# Parse a standard ISO date string
date_str = '2021-05-14'
print('Original string:', date_str, '— type:', type(date_str))

# Convert to full datetime
d2 = datetime.strptime(date_str, '%Y-%m-%d')
print('As datetime:    ', d2, '— type:', type(d2))

# Extract just the date portion (no time)
d3 = datetime.strptime(date_str, '%Y-%m-%d').date()
print('As date:        ', d3, '— type:', type(d3))

Original string: 2021-05-14 — type: <class 'str'>
As datetime:     2021-05-14 00:00:00 — type: <class 'datetime.datetime'>
As date:         2021-05-14 — type: <class 'datetime.date'>


### `strftime` — datetime to string

In [13]:
# Format a datetime object into a custom string representation
d1 = datetime.now()
print('datetime object:', d1)

formatted = d1.strftime('%d/%m/%Y %H:%M')
print('Formatted string:', formatted)
print('Type:', type(formatted))

datetime object: 2026-05-12 12:02:59.036775
Formatted string: 12/05/2026 12:02
Type: <class 'str'>


---

## Part 3: Arithmetic with `timedelta`

When you subtract one `datetime` from another, you get a **`timedelta`** — an object representing the duration between them. This makes it easy to calculate how many days, hours, or minutes separate two events.

In [14]:
# Subtract two datetimes to get the duration between them
d1 = datetime(2020, 4, 23, 11, 13, 10)
d2 = datetime(2021, 4, 23, 12, 13, 10)

duration = d2 - d1
print(type(duration))
print(duration)

<class 'datetime.timedelta'>
365 days, 1:00:00


In [15]:
# Access days and seconds separately
print('Days:   ', duration.days)    # full days
print('Seconds:', duration.seconds)  # remaining seconds within the final day

Days:    365
Seconds: 3600


In [16]:
# Convert the total duration to hours or minutes using timedelta division
print('Duration in hours:  ', duration / timedelta(hours=1))
print('Duration in minutes:', duration / timedelta(minutes=1))
print('Duration in seconds:', duration / timedelta(seconds=1))

Duration in hours:   8761.0
Duration in minutes: 525660.0
Duration in seconds: 31539600.0


In [17]:
# Add or subtract time from a datetime using timedelta
d1 = datetime.now().date()
print("Today:           ", d1)
print("In 2 days:       ", d1 + timedelta(days=2))
print("In 2 weeks:      ", d1 + timedelta(weeks=2))

Today:            2026-05-12
In 2 days:        2026-05-14
In 2 weeks:       2026-05-26


---

## Part 4: DateTime in Pandas

Pandas has its own datetime type called **`Timestamp`** — it is the pandas equivalent of Python's `datetime`. When you convert a column with `pd.to_datetime()`, each value becomes a `Timestamp`.

The key advantage: once a column has the `datetime64` dtype, you can extract any date component using the **`.dt` accessor**.

In [18]:
# Parse a date string directly with pandas
ts = pd.to_datetime('24th of April, 2020')
print(ts)
print(type(ts))  # Timestamp — the pandas equivalent of datetime

2020-04-24 00:00:00
<class 'pandas.Timestamp'>


In [19]:
# pd.to_timedelta() does the same as timedelta but for pandas operations
now = datetime.now()
print('Now:        ', now)
print('Tomorrow:   ', now + pd.to_timedelta(1, unit='D'))
print('Next week:  ', now + pd.to_timedelta(1, unit='W'))

Now:         2026-05-12 12:07:35.224473
Tomorrow:    2026-05-13 12:07:35.224473
Next week:   2026-05-19 12:07:35.224473


### Generating date sequences with `pd.date_range()`

`pd.date_range()` creates an evenly spaced sequence of `Timestamp` values. You specify a **start**, an **end** (or a number of **periods**), and a **frequency**.

In [20]:
# Generate a sequence of daily dates for one calendar month
pd.date_range(start='24/4/2020', end='24/5/2020', freq='D')

DatetimeIndex(['2020-04-24', '2020-04-25', '2020-04-26', '2020-04-27',
               '2020-04-28', '2020-04-29', '2020-04-30', '2020-05-01',
               '2020-05-02', '2020-05-03', '2020-05-04', '2020-05-05',
               '2020-05-06', '2020-05-07', '2020-05-08', '2020-05-09',
               '2020-05-10', '2020-05-11', '2020-05-12', '2020-05-13',
               '2020-05-14', '2020-05-15', '2020-05-16', '2020-05-17',
               '2020-05-18', '2020-05-19', '2020-05-20', '2020-05-21',
               '2020-05-22', '2020-05-23', '2020-05-24'],
              dtype='datetime64[us]', freq='D')

In [21]:
# Generate 10 consecutive minute-intervals from now
# Pandas 3.0 note: use freq='min' for minutes — 'T' (alias for minutes) is deprecated
start_date = datetime.today()
dates_minutes = pd.date_range(start=start_date, periods=10, freq='min')
dates_minutes

DatetimeIndex(['2026-05-12 12:10:07.234229', '2026-05-12 12:11:07.234229',
               '2026-05-12 12:12:07.234229', '2026-05-12 12:13:07.234229',
               '2026-05-12 12:14:07.234229', '2026-05-12 12:15:07.234229',
               '2026-05-12 12:16:07.234229', '2026-05-12 12:17:07.234229',
               '2026-05-12 12:18:07.234229', '2026-05-12 12:19:07.234229'],
              dtype='datetime64[us]', freq='min')

In [22]:
# Generate 10 consecutive daily intervals from now
dates_days = pd.date_range(start=start_date, periods=10, freq='D')
dates_days

DatetimeIndex(['2026-05-12 12:10:07.234229', '2026-05-13 12:10:07.234229',
               '2026-05-14 12:10:07.234229', '2026-05-15 12:10:07.234229',
               '2026-05-16 12:10:07.234229', '2026-05-17 12:10:07.234229',
               '2026-05-18 12:10:07.234229', '2026-05-19 12:10:07.234229',
               '2026-05-20 12:10:07.234229', '2026-05-21 12:10:07.234229'],
              dtype='datetime64[us]', freq='D')

### Creating date features from a DataFrame

In [23]:
# Build a small DataFrame with the two date sequences and a random target column
import random

# Generate 10 random class labels (1, 2, or 3)
random_labels = [random.randint(1, 3) for _ in range(10)]

# Create the DataFrame
df = pd.DataFrame({
    'Start_date': dates_minutes,
    'End_date': dates_days,
    'Target': random_labels
})
df.head()

,Start_date,End_date,Target
0,2026-05-12 12:10:07.234229,2026-05-12 12:10:07.234229,2
1,2026-05-12 12:11:07.234229,2026-05-13 12:10:07.234229,3
2,2026-05-12 12:12:07.234229,2026-05-14 12:10:07.234229,1
3,2026-05-12 12:13:07.234229,2026-05-15 12:10:07.234229,1
4,2026-05-12 12:14:07.234229,2026-05-16 12:10:07.234229,1


In [24]:
# Each date value in the column is a Timestamp object
print(df['Start_date'][0])
print(type(df['Start_date'][0]))

2026-05-12 12:10:07.234229
<class 'pandas.Timestamp'>


In [25]:
# Use the .dt accessor to extract the day from the End_date column
# .dt lets you access datetime properties on an entire column at once
df['Day_of_end_date'] = df['End_date'].dt.day
df.head()

,Start_date,End_date,Target,Day_of_end_date
0,2026-05-12 12:10:07.234229,2026-05-12 12:10:07.234229,2,12
1,2026-05-12 12:11:07.234229,2026-05-13 12:10:07.234229,3,13
2,2026-05-12 12:12:07.234229,2026-05-14 12:10:07.234229,1,14
3,2026-05-12 12:13:07.234229,2026-05-15 12:10:07.234229,1,15
4,2026-05-12 12:14:07.234229,2026-05-16 12:10:07.234229,1,16


In [26]:
# Extract the year from Start_date as another new feature
df['Year_of_start_date'] = df['Start_date'].dt.year
df.head()

,Start_date,End_date,Target,Day_of_end_date,Year_of_start_date
0,2026-05-12 12:10:07.234229,2026-05-12 12:10:07.234229,2,12,2026
1,2026-05-12 12:11:07.234229,2026-05-13 12:10:07.234229,3,13,2026
2,2026-05-12 12:12:07.234229,2026-05-14 12:10:07.234229,1,14,2026
3,2026-05-12 12:13:07.234229,2026-05-15 12:10:07.234229,1,15,2026
4,2026-05-12 12:14:07.234229,2026-05-16 12:10:07.234229,1,16,2026


### Practice: Create more date features

Create at least two more columns from `Start_date` or `End_date`. Try extracting `month`, `hour`, `minute`, `weekday`, or `week of year`. Use the `.dt` accessor.

In [28]:
# YOUR CODE HERE
df['Day_of_start_date'] = df['Start_date'].dt.day
df.head()

,Start_date,End_date,Target,Day_of_end_date,Year_of_start_date,Day_of_start_date
0,2026-05-12 12:10:07.234229,2026-05-12 12:10:07.234229,2,12,2026,12
1,2026-05-12 12:11:07.234229,2026-05-13 12:10:07.234229,3,13,2026,12
2,2026-05-12 12:12:07.234229,2026-05-14 12:10:07.234229,1,14,2026,12
3,2026-05-12 12:13:07.234229,2026-05-15 12:10:07.234229,1,15,2026,12
4,2026-05-12 12:14:07.234229,2026-05-16 12:10:07.234229,1,16,2026,12


---

## Part 5: Working with Dates in a Real Dataset

Datasets almost never arrive with dates already in `datetime64` format. The most common situation is that the date column has been loaded as an **`object`** (string) type, and you need to convert it before any date operations will work.

In [29]:
# Load the Seattle weather dataset
df = pd.read_csv('../data/seattle-weather.csv')
df.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012/01/01,0.0,12.8,5.0,4.7,drizzle
1,2012/01/02,10.9,10.6,2.8,4.5,rain
2,2012/01/03,0.8,11.7,7.2,2.3,rain
3,2012/01/04,20.3,12.2,5.6,4.7,rain
4,2012/01/05,1.3,8.9,2.8,6.1,rain


In [30]:
# Check the column types — the date column will appear as 'object' (string)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1461 non-null   str    
 1   precipitation  1461 non-null   float64
 2   temp_max       1461 non-null   float64
 3   temp_min       1461 non-null   float64
 4   wind           1461 non-null   float64
 5   weather        1461 non-null   str    
dtypes: float64(4), str(2)
memory usage: 68.6 KB


In [31]:
# Confirm the date column is stored as a string
print(df['date'][0])
print(type(df['date'][0]))

2012/01/01
<class 'str'>


In [32]:
# Convert the date column from string to datetime
# Pandas 3.0 note: use df['column'] = ... (bracket notation)
df['date'] = pd.to_datetime(df['date'], format='%Y/%m/%d')

In [33]:
# Verify the conversion was successful — date column should now be datetime64
df.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


In [34]:
# Confirm the dtype changed from object to datetime64
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           1461 non-null   datetime64[us]
 1   precipitation  1461 non-null   float64       
 2   temp_max       1461 non-null   float64       
 3   temp_min       1461 non-null   float64       
 4   wind           1461 non-null   float64       
 5   weather        1461 non-null   str           
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 68.6 KB


In [35]:
# Each value is now a Timestamp — ready for date arithmetic and .dt access
print(df['date'][0])
print(type(df['date'][0]))

2012-01-01 00:00:00
<class 'pandas.Timestamp'>


Now that the column has the right type, you can extract components, filter by date range, resample by month, or plot a time series — all operations that would fail on a string column.

---

## Key Takeaways

- Python's `datetime` module provides `date`, `time`, and `datetime` classes for working with individual date/time values as objects, not strings

- **`strptime()`** parses a date string into a `datetime` object; **`strftime()`** formats a `datetime` back to a string — both use format codes like `%Y`, `%m`, `%d`

- Subtracting two `datetime` objects gives a **`timedelta`**, which represents a duration; divide by `timedelta(hours=1)` etc. to convert to any unit

- Pandas uses the **`Timestamp`** type (equivalent to `datetime`) and stores date columns as `datetime64` dtype

- Use **`pd.to_datetime()`** to convert a string column to datetime

- Use **`df['col'] = pd.to_datetime(...)`** (bracket notation) — attribute-style assignment is deprecated under Pandas 3.0 Copy-on-Write

- Use the **`.dt` accessor** to extract components like `.dt.day`, `.dt.month`, `.dt.weekday`, `.dt.year` from an entire column at once

- Use **`pd.date_range()`** to generate sequences of timestamps; use `freq='min'` for minutes (`freq='T'` is deprecated in Pandas 3.0)

## Check Your Understanding

1. You load a CSV and the `timestamp` column has dtype `object`. What does this tell you, and what should you do before using it in any date calculations?

2. What is the difference between `strptime()` and `strftime()`? When would you use each?

3. You subtract two `datetime` objects and get `365 days, 3600 seconds`. How many hours is the total duration?

4. In Pandas, what does the `.dt` accessor do, and what must be true about the column before you can use it?

5. Why is `df['date'] = pd.to_datetime(df['date'])` preferred over `df.date = pd.to_datetime(df.date)` in Pandas 3.0?